In [ ]:
# import sys
# print(sys.executable)

In [ ]:
%load_ext autoreload
%autoreload 2

# Atari 视频生成实验（Transformer 先验）

在 `Atari_recon` 训练好的 VQ-VAE 基础上：

1. rollout 逐帧 encode → 索引图序列
2. **Action-Conditional Transformer 先验** 学习 `p(z_t | z_{<t}, a_{<=t})`
3. 推理：前 **6** 帧 context + 动作 → 自回归生成后 **10** 帧 → VQ-VAE decode

**待实现模块**（`Video_Ex/`）：
- `GetData.py` → `load_rollout_videos()` · `get_atari_indices_loaders(...)`
- `Model.py` → `load_vqvae_checkpoint(...)` · `TemporalTransformerPrior(...)`
- `Train.py` → `train_video_prior(...)` · `draw_video_strip(...)`

# 1. 导包 + 全局变量

In [ ]:
# import GetData
# import Model
# import Train
#
# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# import os
# from tqdm import tqdm

In [ ]:
# EPOCHS = ...
# PRINT_EPOC = ...
# BATCH_SIZE = ...
# lr = 2e-4
# wd = 1e-5
# device = "cuda:0" if torch.cuda.is_available() else "cpu"
#
# SEQ_LEN = 16
# CONTEXT_LEN = 6
# GENERATE_LEN = 10
# N_ACTIONS = 18
# IMAGE_SIZE = 64
# mean = [0.0, 0.0, 0.0]
# std  = [1.0, 1.0, 1.0]
#
# VQVAE_CKPT = r"..\checkpoints\best_vqvae_atari.pt"
# PRIOR_CKPT = r"..\checkpoints\best_atari_transformer_prior.pt"
# INDICES_CACHE = r"..\checkpoints\Atari_indices_rollouts.pt"
# ACTIONS_CACHE = r"..\checkpoints\Atari_actions_rollouts.pt"

In [ ]:
# --- Transformer 先验超参 ---
# PRIOR_DIM = 256
# PRIOR_LAYERS = 6
# PRIOR_HEADS = 8
# PRIOR_MLP_RATIO = 4.0
# NUM_EMBEDDINGS = 512
# LATENT_SIZE = 16

# 2. 加载 VQ-VAE（冻结）

**接口**：`Model.load_vqvae_checkpoint(path, device)` → `(vqvae_model, checkpoint)`

In [ ]:
# vqvae_model, vqvae_ckpt = Model.load_vqvae_checkpoint(VQVAE_CKPT, device)
# for p in vqvae_model.parameters():
#     p.requires_grad = False

# 3. 编码 rollout → 索引序列

## 3.1 读取视频 rollout

**接口**：`GetData.load_rollout_videos()` → `list[{frames, actions}]`

In [ ]:
# rollouts = GetData.load_rollout_videos()

## 3.2 逐帧 encode 并缓存

每条轨迹 → `indices (T, H, W)` + `actions (T,)`，保存到 `checkpoints/`。

In [ ]:
# vqvae_model.eval()
# with torch.no_grad():
#     for item in rollouts:
#         ... vqvae_model.encode(frame) ...
# torch.save(indices_list, INDICES_CACHE)
# torch.save(actions_list, ACTIONS_CACHE)

## 3.3 构建索引序列 DataLoader

**接口**：`GetData.get_atari_indices_loaders(indices_list, actions_list, seq_len, ...)`

batch: `(z_seq, action_seq)`，形状 `(B, T, H, W)` · `(B, T)`。

In [ ]:
# train_loader, valid_loader = GetData.get_atari_indices_loaders(
#     indices_list, actions_list, seq_len=SEQ_LEN, train_bs=BATCH_SIZE, test_bs=BATCH_SIZE,
# )

# 4. 训练 Transformer 先验

## 4.1 定义模型

**接口**：`Model.TemporalTransformerPrior(...)`

| 方法 | 说明 |
|------|------|
| `forward(indices, actions)` | teacher forcing → logits `(B,T-1,H,W,K)` |
| `generate(context, actions, n_generate, temperature)` | 自回归采样 |

In [ ]:
# prior_model = Model.TemporalTransformerPrior(...)

## 4.2 训练

**接口**：`Train.train_video_prior(prior, train_loader, valid_loader, ...)`

In [ ]:
# prior_model, train_loss_list, valid_loss_list = Train.train_video_prior(
#     prior_model, train_loader, valid_loader, EPOCHS, PRINT_EPOC, lr, wd, device,
# )

## 4.3 损失曲线 · 4.4 保存 checkpoint

→ `checkpoints/best_atari_transformer_prior.pt`

In [ ]:
# plt.plot(train_loss_list, ...); plt.plot(valid_loss_list, ...); plt.show()
# torch.save({"model_state_dict": ..., "config": ...}, PRIOR_CKPT)

# 5. 推理：6 context + 10 generate

对齐论文 Figure 7：潜空间自回归 → 统一 decode。

## 5.1 加载先验

In [ ]:
# prior_Model = Model.TemporalTransformerPrior(...)
# prior_Model.load_state_dict(...)

## 5.2 取 context + 指定动作

In [ ]:
# context = demo_z[:CONTEXT_LEN].unsqueeze(0)   # (1, 6, H, W)
# full_actions = ...                            # (1, 16)

## 5.3 潜空间自回归生成

`prior_Model.generate(context, full_actions, n_generate=GENERATE_LEN, temperature=1.0)`

In [ ]:
# z_full = prior_Model.generate(context, full_actions, GENERATE_LEN)

## 5.4 decode + 可视化

`vqvae_model.decode(z_full[:, t])` · `Train.draw_video_strip(...)`

In [ ]:
# for t in range(z_full.shape[1]):
#     img = vqvae_model.decode(z_full[:, t])
# Train.draw_video_strip(frames_out, n_cols=16, mean=mean, std=std, title="6+10")

## 5.5 （可选）同一 context，不同动作对比

例：全 forward vs 全 right。

In [ ]:
# generate_with_action(context, ACTION_FORWARD, ...)
# generate_with_action(context, ACTION_RIGHT, ...)